# EcoRewind 2.0 — Phase 2: evaluation harness + CPU baselines (Colab, **CPU runtime**)

Persistence, seasonal climatology, harmonic regression, matched DiD / synthetic control and LightGBM, on all folds, both tracks, the real events and the pre-registered placebo windows. Expected run time **2–3 h** (mostly LightGBM), ~0.2 compute units. The run **resumes**: if Colab disconnects, run all cells again and finished parts are skipped.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/ecorewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'

### 1. Tests

In [ ]:
!python -m pytest -q tests

### 2. Cubes on the VM (skipped if already built in this session; ~15 min otherwise)

In [ ]:
if not os.path.isdir('/content/cubes/mississippi_B.zarr'):
    !python scripts/copy_exports.py --src /content/drive/MyDrive/EcoRewind2_exports --dst /content/exports
    !python scripts/build_cube.py --exports /content/exports --out /content/cubes | grep -v ': optical'
!ls /content/cubes

In [ ]:
!python scripts/build_cube.py --exports /content/exports --out /content/cubes --zip-to /content/drive/MyDrive/EcoRewind2/cubes
!ls -lh /content/drive/MyDrive/EcoRewind2/cubes

### 3. Baselines (resumable). Track B first: it is fast and gives the Irma results early.

In [ ]:
RUN = 'phase2_baselines_v1'
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --track B --table-cache /content/tables
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --table-cache /content/tables

In [ ]:
D = '/content/drive/MyDrive/EcoRewind2'
RUN = 'phase2_baselines_v2'
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --track B --table-cache /content/tables
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --table-cache /content/tables

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{D}/runs/{RUN}/summary.md').read())

Send `runs/phase2_baselines_v1/summary.md` (and keep `metrics.csv`, `attribution.csv`, `calibration.csv` on Drive). Then **Runtime → Disconnect and delete runtime**.

In [ ]:
!python scripts/cubes_cache.py --drive /content/drive/MyDrive/EcoRewind2/cubes --local /content/cubes --exports-drive /content/drive/MyDrive/EcoRewind2_exports 2>&1 | grep -v ': optical'
!ls -lh /content/drive/MyDrive/EcoRewind2/cubes

In [ ]:
%cd /content/ecorewind2
!git pull -q
!git log --oneline -1          # should show 66828a9
!python scripts/cubes_cache.py --drive /content/drive/MyDrive/EcoRewind2/cubes --local /content/cubes --exports-drive /content/drive/MyDrive/EcoRewind2_exports